# Integração numérica

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/11-integracao.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. deduzir as regras do trapézio e de Simpson e prever a ordem do erro;
> 2. acelerar a convergência com Romberg;
> 3. deduzir as quadraturas gaussianas, entender por que $n$ pontos bastam para polinômios de grau $2n - 1$ e conferir uma regra de Gauss;
> 4. entender o que uma rotina adaptativa como `scipy.integrate.quad` faz, e quando ela erra calada;
> 5. tratar integrandos singulares com mudança de variável;
> 6. usar Monte Carlo em muitas dimensões, com a barra de erro certa.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
from scipy import integrate
from cnlivro.integracao import (trapezio, simpson, trapezio_dados, gauss_legendre, romberg,
                                simpson_adaptativo, monte_carlo)

## 1. Problema de engenharia: quanto tempo para esvaziar um tanque?

Um tanque de decantação tem a forma de um tronco de cone com fundo plano: raio $r_0 = 1$ m no fundo, $r_1 = 2$ m no topo e altura $H = 3$ m. Entre duas bateladas, ele é esvaziado por um orifício no fundo, de diâmetro $d = 5$ cm e coeficiente de descarga $C_d = 0{,}61$. A operação precisa saber quanto tempo isso leva, partindo do tanque cheio.

Pela lei de Torricelli, a vazão pelo orifício é $Q = C_d\, a \sqrt{2gh}$, com $a = \pi d^2/4$ e $h$ o nível da água. O volume que sai é o que o nível desce: $A(h)\,dh = -Q\,dt$, com $A(h)$ a área da seção na altura $h$. Separando as variáveis,

$$
T = \int_0^H \frac{A(h)}{C_d\, a\sqrt{2g}}\,\frac{dh}{\sqrt{h}},
\qquad A(h) = \pi\left(r_0 + (r_1 - r_0)\frac{h}{H}\right)^2 .
$$ (eq-tanque)

Para essa forma há primitiva, e o resultado exato servirá para conferir. Para um tanque de forma qualquer, dada por uma **tabela de arqueação** (área em função do nível, medida na obra), só resta integrar numericamente. Repare no fator $1/\sqrt{h}$: o integrando é **infinito** em $h = 0$, embora a integral seja finita.

In [ ]:
g, r0, r1, H = 9.81, 1.0, 2.0, 3.0          # m/s², m, m, m
d, Cd = 0.05, 0.61                          # m, adimensional
k = Cd * np.pi * d**2 / 4 * np.sqrt(2 * g)  # m^(5/2)/s

def area(h):
    return np.pi * (r0 + (r1 - r0) * h / H) ** 2

def integrando(h):
    return area(h) / (k * np.sqrt(h))

# primitiva de π (r0 + c h)² / √h, termo a termo
c = (r1 - r0) / H
T_exato = np.pi / k * (2 * r0**2 * H**0.5 + 4 / 3 * r0 * c * H**1.5 + 2 / 5 * c**2 * H**2.5)
print(f"tempo exato de esvaziamento: {T_exato:.1f} s = {T_exato / 60:.2f} min")

In [ ]:
hs = np.linspace(1e-3, H, 600)
fig = go.Figure()
fig.add_trace(go.Scatter(x=hs, y=integrando(hs), mode="lines", fill="tozeroy", name="A(h) / (C_d a √(2g h))",
                         line=dict(color=CORES["aproximacao"]), fillcolor="rgba(0,114,178,0.15)"))
fig.update_xaxes(title="nível h (m)")
fig.update_yaxes(title="integrando (s/m)", range=[0, 6000])
fig.update_layout(width=760, height=400, title="A área sob a curva é o tempo de esvaziamento", showlegend=False)
mostrar(fig)

**O que notar:** perto do topo, o integrando é quase constante. Perto do fundo, ele dispara para o infinito: com pouca água, a pressão é pequena, a vazão também, e cada centímetro de nível demora muito. A área sob esse pico é finita, mas uma regra que avalie o integrando em $h = 0$ tem um problema.

## 2. A ideia em uma imagem

Integrar é calcular a área sob uma curva. As regras de Newton-Cotes trocam a curva, em cada pedaço, por algo que sabemos integrar: retas (trapézio) ou parábolas (Simpson). Arraste o slider para aumentar o número de subintervalos $n$.

In [ ]:
f2 = lambda x: 1.2 + 0.8 * np.sin(2.2 * x) * np.exp(-0.25 * x) + 0.15 * x
a2, b2 = 0.0, 4.0
exato2 = integrate.quad(f2, a2, b2)[0]
xx = np.linspace(a2, b2, 400)

def quadro_nc(n):
    # títulos fixos nos painéis; os erros vão no título geral, que muda com o slider
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=["trapézio", "Simpson"])
    xn = np.linspace(a2, b2, n + 1)
    # trapézio: polígono pelos nós
    fig.add_trace(go.Scatter(x=np.r_[xn, xn[::-1]], y=np.r_[f2(xn), np.zeros(n + 1)], fill="toself", mode="lines",
                             fillcolor="rgba(213,94,0,0.18)", line=dict(color=CORES["erro"], width=1.5),
                             name="aproximação"), 1, 1)
    # Simpson: parábola por três nós em cada par de subintervalos
    xs_p, ys_p = [], []
    for i in range(0, n, 2):
        xa, xm, xb = xn[i], xn[i + 1], xn[i + 2]
        p = np.polyfit([xa, xm, xb], f2(np.array([xa, xm, xb])), 2)
        xl = np.linspace(xa, xb, 40)
        xs_p += list(xl)
        ys_p += list(np.polyval(p, xl))
    xs_p, ys_p = np.array(xs_p), np.array(ys_p)
    fig.add_trace(go.Scatter(x=np.r_[xs_p, xs_p[::-1]], y=np.r_[ys_p, np.zeros_like(ys_p)], fill="toself", mode="lines",
                             fillcolor="rgba(213,94,0,0.18)", line=dict(color=CORES["erro"], width=1.5),
                             showlegend=False), 1, 2)
    for col in (1, 2):
        fig.add_trace(go.Scatter(x=xx, y=f2(xx), mode="lines", line=dict(color=CORES["exata"], width=2),
                                 name="f(x)", showlegend=(col == 1)), 1, col)
        fig.add_trace(go.Scatter(x=xn, y=f2(xn), mode="markers", marker=dict(color=CORES["destaque"], size=8),
                                 name="nós", showlegend=(col == 1)), 1, col)
    fig.update_xaxes(title="x")
    fig.update_yaxes(range=[0, 2.6])
    fig.update_layout(width=980, height=450, legend=dict(orientation="h", y=-0.2), margin=dict(t=110),
                      title=f"n = {n}: erro do trapézio {numero(trapezio(f2, a2, b2, n) - exato2, 4)}, "
                            f"erro de Simpson {numero(simpson(f2, a2, b2, n) - exato2, 4)}")
    return fig

quadros = [(str(n), quadro_nc(n)) for n in [2, 4, 8, 16]]
mostrar(com_controle(quadros, rotulo="n = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com o mesmo número de nós, as parábolas acompanham a curva muito melhor que as retas. Dobrar $n$ divide o erro do trapézio por cerca de 4, e o de Simpson por cerca de 16. Com $n = 16$, a aproximação de Simpson já é indistinguível da curva.

## 3. Formulação matemática

### 3.1 Regras de Newton-Cotes

Dividimos $[a, b]$ em $n$ subintervalos de largura $h = (b - a)/n$, com nós $x_i = a + ih$ e valores $f_i = f(x_i)$.

**Trapézio.** Em cada subintervalo, a área de um trapézio. Somando,

$$
T_n = h\left[\frac{f_0}{2} + f_1 + \cdots + f_{n-1} + \frac{f_n}{2}\right],
\qquad
\int_a^b f\,dx - T_n = -\frac{(b - a)\,h^2}{12}f''(\xi).
$$ (eq-trapezio)

O erro é de **ordem 2** e exato para retas.

**Simpson 1/3.** Uma parábola por cada trio de nós (por isso $n$ precisa ser **par**):

$$
S_n = \frac{h}{3}\left[f_0 + 4f_1 + 2f_2 + 4f_3 + \cdots + 4f_{n-1} + f_n\right],
\qquad
\int_a^b f\,dx - S_n = -\frac{(b - a)\,h^4}{180}f^{(4)}(\xi).
$$ (eq-simpson)

O erro é de **ordem 4**. Por sorte de simetria, a regra é exata para cúbicas, e não só para parábolas. A regra 3/8 de Simpson usa cúbicas por quatro nós, com a mesma ordem; serve para completar um número ímpar de subintervalos.

**Dados tabelados.** Com espaçamento irregular, o trapézio continua simples: $\sum_i (x_{i+1} - x_i)(y_i + y_{i+1})/2$, que é `np.trapezoid(y, x)`. Para Simpson com espaçamento irregular, `scipy.integrate.simpson(y, x=x)`.

**Integrar uma spline.** Se os dados já foram interpolados por uma spline (capítulo 6), a integral da spline é **exata** e barata: em cada intervalo ela é um polinômio, que se integra termo a termo, e o custo é um termo por intervalo (`CubicSpline(x, y).integrate(a, b)`). O erro que sobra é só o da interpolação, $O(h^4)$ para a cúbica. Na base de B-splines (seção 3.7 do capítulo 6) a conta é ainda mais direta, porque $\int B_{i,k}\,dx = (t_{i+k+1} - t_i)/(k + 1)$ e então $\int S\,dx = \sum_i c_i\,(t_{i+k+1} - t_i)/(k + 1)$. Com $k = 1$, isso é a regra do trapézio: integrar exatamente a interpolação linear dos dados.

As fórmulas de erro só valem se as derivadas que aparecem nelas **existirem e forem limitadas**. Em {eq}`eq-tanque`, $f''$ e $f^{(4)}$ são infinitas em $h = 0$, e a ordem prometida não se realiza (seção 6).

### 3.2 Romberg: Richardson aplicado ao trapézio

Para $f$ suave, o erro do trapézio tem a forma $c_2 h^2 + c_4 h^4 + c_6 h^6 + \cdots$, só com potências pares (fórmula de Euler-Maclaurin). Como no capítulo 10, combinando $T_n$ e $T_{2n}$ elimina-se o primeiro termo:

$$
R_{k,1} = \frac{4T_{2n} - T_n}{3}, \qquad
R_{k,j} = R_{k,j-1} + \frac{R_{k,j-1} - R_{k-1,j-1}}{4^j - 1}.
$$

A primeira coluna extrapolada é exatamente a regra de Simpson. Cada nível dobra os nós, reaproveitando os anteriores, e ganha duas ordens.


### 3.3 Quadraturas gaussianas

#### A pergunta certa: onde avaliar $f$?

Todas as regras deste capítulo têm a mesma forma, uma **soma ponderada** de valores de $f$:

$$
\int_a^b f(x)\,dx \approx \sum_{i=1}^{n} w_i\, f(x_i).
$$ (eq-quadratura)

Uma regra fica definida por dois conjuntos de números: os **nós** $x_i$, onde $f$ é avaliada, e os **pesos** $w_i$, que dizem quanto cada valor conta. Para comparar regras, usa-se o **grau de exatidão**: o maior grau $m$ tal que a regra acerta **exatamente** a integral de todo polinômio de grau até $m$. Esse número importa porque, num pedaço pequeno, uma função suave é quase igual ao seu polinômio de Taylor. Se a regra integra sem erro o polinômio de grau $m$, o único erro que sobra vem do resto de Taylor, proporcional a $h^{m+1}$ em cada pedaço. Mais grau de exatidão significa erro menor.

**Quantos graus de liberdade temos?** Nas regras de Newton-Cotes, os nós são **impostos**: igualmente espaçados, incluindo as pontas. Só os $n$ pesos são escolhidos, e com $n$ incógnitas podemos impor $n$ condições, ou seja, acertar $1, x, \dots, x^{n-1}$. O grau de exatidão é $n - 1$ (às vezes $n$, por sorte de simetria, como em Simpson).

A ideia de Gauss (1814) é deixar os nós **livres também**. Com $n$ nós e $n$ pesos, são $2n$ incógnitas, e podemos impor $2n$ condições: acertar $1, x, x^2, \dots, x^{2n-1}$. O grau de exatidão sobe para $2n - 1$, quase o dobro, **com o mesmo número de avaliações de $f$**. Como o custo de uma integral é justamente o número de avaliações (seção 5.1), é precisão de graça.

| Regra | avaliações de $f$ | grau de exatidão |
| --- | --- | --- |
| trapézio | 2 | 1 |
| Simpson 1/3 | 3 | 3 |
| Simpson 3/8 | 4 | 3 |
| Gauss com 2 pontos | 2 | 3 |
| Gauss com 3 pontos | 3 | 5 |
| Gauss com $n$ pontos | $n$ | $2n - 1$ |

E não dá para fazer melhor. Para quaisquer $n$ nós, o polinômio $p(x) = (x - x_1)^2 (x - x_2)^2 \cdots (x - x_n)^2$, de grau $2n$, é positivo fora dos nós, então sua integral é positiva; mas ele se anula em todos os nós, e a regra devolve zero. Nenhuma regra com $n$ pontos tem grau $2n$.

A figura mostra a ideia com uma cúbica em $[-1, 1]$. As duas regras usam **dois** valores de $f$ e, na prática, integram uma reta. O trapézio passa a reta pelas pontas; Gauss passa a reta por dois pontos internos escolhidos de modo que as áreas que sobram e que faltam se **cancelem exatamente**.

In [ ]:
fc = lambda t: 0.5 * t**3 - 0.8 * t**2 + 0.2 * t + 1.8     # uma cúbica qualquer
exato_c = 2 * 1.8 - 0.8 * 2 / 3                             # integral exata em [-1, 1]
tt = np.linspace(-1, 1, 400)
tg = 1 / np.sqrt(3)

def reta_por(t1, t2):
    inc = (fc(t2) - fc(t1)) / (t2 - t1)
    return lambda t: fc(t1) + inc * (t - t1)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=["trapézio: reta pelas pontas", "Gauss: reta pelos nós ±1/√3"])
for col, (t1, t2) in [(1, (-1.0, 1.0)), (2, (-tg, tg))]:
    reta = reta_por(t1, t2)
    # área entre a curva e a reta: laranja onde falta (curva acima), azul onde sobra (reta acima)
    acima = np.maximum(fc(tt), reta(tt))
    fig.add_trace(go.Scatter(x=tt, y=reta(tt), mode="lines", line=dict(width=0), showlegend=False,
                             hoverinfo="skip"), 1, col)
    fig.add_trace(go.Scatter(x=tt, y=np.maximum(fc(tt), reta(tt)), mode="lines", line=dict(width=0),
                             fill="tonexty", fillcolor="rgba(213,94,0,0.30)", name="área que a reta perde",
                             showlegend=(col == 1), hoverinfo="skip"), 1, col)
    fig.add_trace(go.Scatter(x=tt, y=fc(tt), mode="lines", line=dict(width=0), showlegend=False,
                             hoverinfo="skip"), 1, col)
    fig.add_trace(go.Scatter(x=tt, y=np.maximum(fc(tt), reta(tt)), mode="lines", line=dict(width=0),
                             fill="tonexty", fillcolor="rgba(0,114,178,0.30)", name="área que a reta ganha",
                             showlegend=(col == 1), hoverinfo="skip"), 1, col)
    fig.add_trace(go.Scatter(x=tt, y=fc(tt), mode="lines", line=dict(color=CORES["exata"], width=2.5),
                             name="f(t), cúbica", showlegend=(col == 1)), 1, col)
    fig.add_trace(go.Scatter(x=tt, y=reta(tt), mode="lines", line=dict(color=CORES["aproximacao"], width=2,
                             dash="dash"), name="reta da regra", showlegend=(col == 1)), 1, col)
    fig.add_trace(go.Scatter(x=[t1, t2], y=fc(np.array([t1, t2])), mode="markers",
                             marker=dict(color=CORES["destaque"], size=11), name="nós (onde f é avaliada)",
                             showlegend=(col == 1)), 1, col)
trap_c = fc(-1.0) + fc(1.0)
gauss_c = fc(-tg) + fc(tg)
fig.update_xaxes(title="t", range=[-1.05, 1.05])
fig.update_yaxes(range=[0, 2.2])
fig.update_layout(width=980, height=460, margin=dict(t=110), legend=dict(orientation="h", y=-0.22),
                  title=f"Mesmas duas avaliações: erro do trapézio {numero(exato_c - trap_c, 3)}, "
                        f"erro de Gauss {numero(abs(exato_c - gauss_c), 3)}")
mostrar(fig)

**O que notar:** no trapézio, a reta pelas pontas fica abaixo da cúbica em quase todo o intervalo, e a área que falta não é compensada. Na regra de Gauss, a reta corta a cúbica em dois pontos internos: a área que ela perde no meio é exatamente a que ganha perto das pontas. Isso vale para **qualquer** cúbica, não só para esta. É o que significa grau de exatidão 3 com apenas dois nós.

#### Gauss com 2 pontos, deduzida

Trabalhamos no intervalo padrão $[-1, 1]$; a passagem para um $[a, b]$ qualquer vem logo adiante. Queremos que $w_1 f(t_1) + w_2 f(t_2)$ seja exata para $f = 1, t, t^2, t^3$. Como $\int_{-1}^{1} t^k\,dt$ vale $2/(k+1)$ para $k$ par e $0$ para $k$ ímpar, as quatro condições são

$$
\begin{aligned}
f = 1:&\quad w_1 + w_2 = 2,\\
f = t:&\quad w_1 t_1 + w_2 t_2 = 0,\\
f = t^2:&\quad w_1 t_1^2 + w_2 t_2^2 = \tfrac{2}{3},\\
f = t^3:&\quad w_1 t_1^3 + w_2 t_2^3 = 0.
\end{aligned}
$$

É um sistema **não linear** (os nós aparecem em potências), mas a simetria o resolve. O intervalo é simétrico em torno de $0$: se trocarmos $t$ por $-t$, as integrais não mudam, e a regra refletida $(-t_i, w_i)$ também satisfaz as quatro equações. Como a solução é única, ela é a própria refletida: $t_2 = -t_1$ e $w_2 = w_1$. Com isso, as equações ímpares ficam automaticamente satisfeitas, a primeira dá $w_1 = w_2 = 1$ e a terceira dá $2t_1^2 = 2/3$:

$$
\int_{-1}^{1} f(t)\,dt \approx f\!\left(-\frac{1}{\sqrt3}\right) + f\!\left(\frac{1}{\sqrt3}\right),
\qquad \frac{1}{\sqrt3} \approx 0{,}577350.
$$ (eq-gauss2)

**Conferência.** Para $f = t^4$, a regra dá $2\,(1/3)^2 = 2/9$, e a integral exata é $2/5$. Não bate: o grau de exatidão é exatamente 3.

#### Gauss com 3 pontos, deduzida

Pela mesma simetria, os nós são $-\tau, 0, \tau$ e os pesos são $A, B, A$. As potências ímpares se integram a zero sozinhas; sobram as condições para $1$, $t^2$ e $t^4$:

$$
2A + B = 2, \qquad 2A\tau^2 = \frac{2}{3}, \qquad 2A\tau^4 = \frac{2}{5}.
$$

Dividindo a terceira pela segunda, $\tau^2 = 3/5$. Daí $A = 1/(3\tau^2) = 5/9$ e $B = 2 - 2A = 8/9$:

$$
\int_{-1}^{1} f(t)\,dt \approx \frac{5}{9} f\!\left(-\sqrt{\tfrac{3}{5}}\right) + \frac{8}{9} f(0) + \frac{5}{9} f\!\left(\sqrt{\tfrac{3}{5}}\right),
\qquad \sqrt{\tfrac{3}{5}} \approx 0{,}774597.
$$ (eq-gauss3)

A regra é exata até o grau 5 (o grau 5 é ímpar e sai de graça pela simetria). Para $t^6$, ela dá $2 \cdot \frac{5}{9} \cdot \left(\frac{3}{5}\right)^3 = \frac{6}{25}$, contra $\frac{2}{7}$ exato. Repare que os pesos são **positivos** e somam $2$, o comprimento do intervalo (é a condição para $f = 1$), e que o nó central pesa mais.

#### O caso geral: as raízes dos polinômios de Legendre

Para $n = 4, 5, \dots$, o sistema não linear deixa de ser resolvível à mão. O atalho de Gauss passa pelos **polinômios de Legendre** $P_n$, de grau $n$, que são **ortogonais** em $[-1, 1]$:

$$
\int_{-1}^{1} P_m(t)\,P_n(t)\,dt = 0 \quad (m \ne n),
\qquad
P_0 = 1,\quad P_1 = t,\quad (k+1)P_{k+1} = (2k+1)\,t\,P_k - k\,P_{k-1}.
$$ (eq-legendre)

Pela recorrência, $P_2 = (3t^2 - 1)/2$ e $P_3 = (5t^3 - 3t)/2$. As raízes de $P_2$ são $\pm 1/\sqrt3$, e as de $P_3$ são $0$ e $\pm\sqrt{3/5}$: exatamente os nós que deduzimos. Não é coincidência. Como $P_n$ é ortogonal a $P_0, \dots, P_{n-1}$, ele é ortogonal a **todo** polinômio de grau menor que $n$.

> **Teorema.** Se os nós $t_i$ são as $n$ raízes de $P_n$ e os pesos são escolhidos para que a regra seja exata até o grau $n - 1$, então ela é exata até o grau $2n - 1$.
>
> **Prova.** Seja $p$ um polinômio de grau até $2n - 1$. Dividindo $p$ por $P_n$, temos $p = q\,P_n + r$, com o quociente $q$ e o resto $r$ de grau até $n - 1$. A integral é $\int p = \int q\,P_n + \int r = \int r$, porque $q$ tem grau menor que $n$. A regra dá $\sum w_i\, p(t_i) = \sum w_i\, q(t_i) P_n(t_i) + \sum w_i\, r(t_i) = \sum w_i\, r(t_i)$, porque $P_n(t_i) = 0$. E $\sum w_i\, r(t_i) = \int r$, porque $r$ tem grau até $n - 1$. Logo a regra acerta $\int p$. $\square$

Os pesos têm fórmula fechada, $w_i = \dfrac{2}{(1 - t_i^2)\,[P_n'(t_i)]^2}$, e a teoria garante duas propriedades que importam na prática:

1. **As $n$ raízes são reais, distintas e ficam dentro de $(-1, 1)$.** A regra nunca avalia $f$ nas pontas, o que permite aplicá-la (com cuidado, seção 7.4) a integrandos infinitos numa ponta, como o do tanque.
2. **Todos os pesos são positivos e somam 2.** Se cada valor $f(t_i)$ tem um erro $\varepsilon_i$ (arredondamento, ruído), o erro na soma é no máximo $2\max|\varepsilon_i|$: a regra é **estável** para qualquer $n$. As regras de Newton-Cotes de grau alto não têm essa garantia: a partir de 9 nós aparecem pesos negativos, que amplificam os erros, e por isso ninguém as usa.

Na prática, ninguém resolve o sistema não linear nem procura raízes ao acaso. A recorrência {eq}`eq-legendre` define uma matriz tridiagonal simétrica (a matriz de Jacobi) cujos **autovalores são os nós** e cujos autovetores dão os pesos: é o algoritmo de Golub-Welsch, uma aplicação direta do capítulo 8. O NumPy ainda refina as raízes com um passo de Newton (capítulo 3).

#### Mudança de intervalo

Para integrar em $[a, b]$, troca-se a variável com uma função afim que leva $[-1, 1]$ em $[a, b]$:

$$
x = \frac{a + b}{2} + \frac{b - a}{2}\,t, \qquad dx = \frac{b - a}{2}\,dt,
\qquad
\int_a^b f(x)\,dx \approx \frac{b - a}{2} \sum_{i=1}^{n} w_i\, f\!\left(\frac{a + b}{2} + \frac{b - a}{2}\,t_i\right).
$$ (eq-gauss-ab)

Os nós ficam nas mesmas **posições relativas** (o centro do intervalo mais uma fração do meio comprimento), e os pesos são multiplicados pelo meio comprimento $(b - a)/2$, de modo que passam a somar $b - a$. Uma função afim leva polinômio de grau $m$ em polinômio de grau $m$, então o grau de exatidão continua $2n - 1$. Em $[0, 1]$, por exemplo, os dois nós de Gauss ficam em $0{,}5 \pm 0{,}5/\sqrt3$, com peso $0{,}5$ cada (seção 4).

#### O erro, e as duas maneiras de reduzi-lo

Se $f$ tem $2n$ derivadas contínuas em $[a, b]$,

$$
\int_a^b f\,dx - G_n = \frac{(b - a)^{2n+1}\,(n!)^4}{(2n + 1)\,[(2n)!]^3}\, f^{(2n)}(\xi).
$$ (eq-erro-gauss)

Para $n = 2$, o fator é $(b - a)^5/4320$; o de Simpson simples, que usa **três** avaliações, é $(b - a)^5/2880$. Gauss com 2 pontos tem a mesma ordem de Simpson, erro cerca de 2/3 do dele, e uma avaliação a menos. Para $n = 3$, o fator já é $(b - a)^7/2\,016\,000$.

O denominador cresce como um fatorial. Para uma função **analítica** (com série de Taylor convergente perto de $[a, b]$), o erro cai **exponencialmente** com $n$, mais depressa que qualquer potência (seção 5.1). Há duas maneiras de reduzir o erro:

- **aumentar $n$** num intervalo só (refinamento $p$): o melhor caminho quando $f$ é suave em todo o intervalo;
- **dividir $[a, b]$ em $m$ subintervalos** com Gauss de $n$ pontos em cada um (**Gauss composto**, refinamento $h$): o erro cai como $h^{2n}$, com $h = (b - a)/m$. É o caminho robusto quando $f$ tem regiões difíceis, porque $f^{(2n)}$ só precisa ser limitada em cada pedaço.

Uma desvantagem: os nós de $G_n$ e de $G_{n+1}$ não coincidem, e os nós de Gauss composto não se repetem quando os subintervalos são divididos ao meio. Ao contrário de Romberg, refinar Gauss **não reaproveita** avaliações anteriores. A solução é a regra de Gauss-Kronrod (seção 3.4).

#### Outras quadraturas gaussianas

A mesma construção funciona com um **peso** $\rho(x) > 0$ embutido na integral: $\int \rho(x) f(x)\,dx \approx \sum w_i f(x_i)$, com os nós nas raízes dos polinômios ortogonais em relação a $\rho$. A regra fica exata para $\rho$ vezes um polinômio de grau $2n - 1$, e a parte difícil do integrando (uma singularidade, um intervalo infinito) fica por conta do peso.

| Família | intervalo | peso $\rho(x)$ | uso típico | NumPy |
| --- | --- | --- | --- | --- |
| Gauss-Legendre | $[-1, 1]$ | $1$ | integrais comuns | `legendre.leggauss` |
| Gauss-Chebyshev | $[-1, 1]$ | $1/\sqrt{1 - x^2}$ | integrandos infinitos nas duas pontas | `chebyshev.chebgauss` |
| Gauss-Laguerre | $[0, \infty)$ | $e^{-x}$ | integrandos com decaimento exponencial | `laguerre.laggauss` |
| Gauss-Hermite | $(-\infty, \infty)$ | $e^{-x^2}$ | médias sob a distribuição normal | `hermite.hermgauss` |

Gauss-Hermite aparece na propagação de incertezas (apêndice A): se $X$ é normal com média $\mu$ e desvio $\sigma$, então $\mathrm{E}[g(X)] \approx \frac{1}{\sqrt\pi}\sum_i w_i\, g(\mu + \sqrt2\,\sigma\, t_i)$, com poucos pontos em vez de milhares de sorteios. Há ainda variantes que incluem as pontas entre os nós: **Gauss-Lobatto** (as duas pontas, grau $2n - 3$) e **Gauss-Radau** (uma ponta, grau $2n - 2$), que estão por trás do método implícito Radau IIA para EDOs rígidas (capítulo 13).

#### Na prática

Os nós e pesos já estão tabelados nas bibliotecas, com precisão de máquina, para qualquer $n$:

- `t, w = np.polynomial.legendre.leggauss(n)` devolve nós e pesos em $[-1, 1]$; a mudança de intervalo {eq}`eq-gauss-ab` fica por sua conta;
- `scipy.integrate.fixed_quad(f, a, b, n=n)` faz tudo numa linha (com `f` vetorizada), sem estimativa de erro;
- `scipy.special.roots_legendre(n)` dá os mesmos nós e pesos, e funciona bem até para $n$ na casa dos milhares;
- `scipy.integrate.quad(f, a, b)` usa Gauss-Kronrod adaptativo, com estimativa de erro: é a escolha padrão quando não se sabe nada sobre $f$;
- em PyTorch não há uma função pronta, mas basta converter `t` e `w` em tensores: a soma ponderada $\sum w_i f(x_i)$ é diferenciável em relação a parâmetros de $f$, o que serve para colocar integrais dentro de funções de perda (capítulo 9).

As quatro primeiras devem concordar até o arredondamento:

In [ ]:
f_ex = lambda x: np.exp(-x**2)
a_ex, b_ex, n_ex = 0.0, 1.0, 5
I_ref = integrate.quad(f_ex, a_ex, b_ex, epsabs=1e-14)[0]

t, w = np.polynomial.legendre.leggauss(n_ex)               # nós e pesos em [-1, 1]
x = (a_ex + b_ex) / 2 + (b_ex - a_ex) / 2 * t              # mudança de intervalo
manual = (b_ex - a_ex) / 2 * np.sum(w * f_ex(x))

for nome, valor in [("leggauss + mudança de intervalo", manual),
                    ("scipy.integrate.fixed_quad", integrate.fixed_quad(f_ex, a_ex, b_ex, n=n_ex)[0]),
                    ("cnlivro gauss_legendre", gauss_legendre(f_ex, a_ex, b_ex, n_ex)),
                    ("scipy.integrate.quad (referência)", I_ref)]:
    print(f"{nome:34s} {valor:.15f}   diferença {valor - I_ref:+.1e}")

Com cinco avaliações de $f$, o erro já é menor que $10^{-8}$.

#### Verificação: como conferir uma regra de Gauss

Se um agente escreveu o código que calcula nós e pesos (ou que faz a mudança de intervalo), não é preciso ler o código linha a linha para confiar nele. A teoria dá três testes baratos, que valem para qualquer $n$ e qualquer intervalo:

1. **Os pesos são positivos e somam $b - a$.** É a exatidão para $f = 1$.
2. **O teste dos monômios.** A regra acerta $\int_a^b x^k\,dx = (b^{k+1} - a^{k+1})/(k+1)$ até o arredondamento para $k = 0, 1, \dots, 2n - 1$, e **erra visivelmente** em $k = 2n$. Acertar em $2n$ também seria um sinal de problema: o teste não estaria testando nada.
3. **A comparação com uma biblioteca** (`fixed_quad` ou `leggauss`).

O teste dos monômios é o mais importante, porque é a própria definição da regra:

In [ ]:
def teste_monomios(regra, n, a, b):
    # erro relativo da regra para x^k, k = 0, ..., 2n + 1
    for k in range(2 * n + 2):
        exato = (b ** (k + 1) - a ** (k + 1)) / (k + 1)
        erro = abs(regra(lambda x: x**k, a, b, n) - exato) / abs(exato)
        marca = "exata" if erro < 1e-13 else "ERRA"
        print(f"  x^{k:<2d}: erro relativo {erro:.1e}  {marca}")

n_t, a_t, b_t = 3, 0.5, 2.0
t, w = np.polynomial.legendre.leggauss(n_t)
w_ab = (b_t - a_t) / 2 * w
print(f"pesos em [{a_t}, {b_t}]: {np.round(w_ab, 6)}, todos positivos: {bool(np.all(w_ab > 0))}, "
      f"soma = {w_ab.sum():.15f} (b − a = {b_t - a_t})")
print(f"teste dos monômios, Gauss com {n_t} pontos (grau esperado 2n − 1 = {2 * n_t - 1}):")
teste_monomios(gauss_legendre, n_t, a_t, b_t)

A regra acerta até $x^5$ e passa a errar em $x^6$, como a teoria prevê para $n = 3$. Um erro comum em código gerado por agente é esquecer o fator $(b - a)/2$ ou escrever a mudança de intervalo com o sinal trocado: o teste falha já em $x^0$ ou $x^1$, e o defeito aparece em uma linha de saída.

### 3.4 Integração adaptativa

Funções reais costumam ser suaves em quase todo o intervalo, com poucas regiões difíceis. Uma rotina **adaptativa** estima o erro em cada pedaço e só subdivide onde é preciso. No Simpson adaptativo, compara-se $S$ no pedaço inteiro com $S$ nas duas metades: pela fórmula {eq}`eq-simpson`, o erro da soma das metades é cerca de $|S_{\text{metades}} - S_{\text{inteiro}}|/15$. Se for maior que a tolerância, cada metade é tratada da mesma forma.

`scipy.integrate.quad` usa a biblioteca QUADPACK: regras de Gauss-Kronrod de 21 pontos, subdivisão adaptativa e extrapolação, que também lida bem com singularidades nas pontas do tipo $1/\sqrt{x}$. É a escolha padrão para funções que você pode avaliar.

### 3.5 Monte Carlo e a maldição da dimensão

Uma regra de produto com $n$ pontos por eixo, em $d$ dimensões, usa $N = n^d$ pontos. Com erro $O(n^{-p})$ por eixo, o erro em função do custo é $O(N^{-p/d})$: em dimensão alta, nenhuma ordem $p$ salva.

**Monte Carlo** sorteia $N$ pontos uniformes numa região de volume $V$ e usa a média:

$$
\int_\Omega f\,d\mathbf{x} \approx V\,\bar f, \qquad \text{erro padrão} \approx \frac{V\,s_f}{\sqrt{N}},
$$ (eq-mc)

com $s_f$ o desvio padrão amostral dos valores de $f$. O erro cai como $N^{-1/2}$ **em qualquer dimensão**. É lento em uma dimensão e imbatível em dez.

## 4. À mão

Vamos calcular $I = \int_0^1 e^{-x^2}\,dx = 0{,}746824$, uma integral sem primitiva elementar.

| $x$ | $0$ | $0{,}25$ | $0{,}5$ | $0{,}75$ | $1$ |
| --- | --- | --- | --- | --- | --- |
| $e^{-x^2}$ | $1$ | $0{,}939413$ | $0{,}778801$ | $0{,}569783$ | $0{,}367879$ |

- **Trapézio, $n = 1$:** $\frac{1}{2}(1 + 0{,}367879) = 0{,}683940$. Erro de $0{,}063$.
- **Trapézio, $n = 2$:** $0{,}5\left[\frac{1}{2} + 0{,}778801 + \frac{0{,}367879}{2}\right] = 0{,}731370$. Erro de $0{,}015$: dividido por 4.
- **Simpson, $n = 2$:** $\frac{0{,}5}{3}(1 + 4 \times 0{,}778801 + 0{,}367879) = 0{,}747180$. Erro de $0{,}00036$.
- **Romberg:** $(4 \times 0{,}731370 - 0{,}683940)/3 = 0{,}747180$. É o mesmo número de Simpson, como prometido na seção 3.2.
- **Gauss com 2 pontos:** nós $0{,}5 \pm 0{,}5/\sqrt3 = 0{,}211325$ e $0{,}788675$, com pesos $0{,}5$: $0{,}5(0{,}956355 + 0{,}536837) = 0{,}746595$. Erro de $0{,}00023$, **menor que o de Simpson com só 2 avaliações** em vez de 3.
- **Gauss com 3 pontos:** em $[0, 1]$, os nós de {eq}`eq-gauss3` ficam em $0{,}5$ e $0{,}5 \pm 0{,}5\sqrt{0{,}6} = 0{,}112702$ e $0{,}887298$, com pesos $\frac{5}{9} \cdot \frac12 = \frac{5}{18}$ e $\frac{8}{9} \cdot \frac12 = \frac{8}{18}$: $\frac{1}{18}(5 \times 0{,}987379 + 8 \times 0{,}778801 + 5 \times 0{,}455073) = 0{,}746815$. Erro de $0{,}00001$: **37 vezes menor que o de Simpson**, com as mesmas 3 avaliações.

In [ ]:
f_hand = lambda x: np.exp(-x**2)
I_ref = integrate.quad(f_hand, 0, 1)[0]
for nome, valor in [("trapézio n=1", trapezio(f_hand, 0, 1, 1)), ("trapézio n=2", trapezio(f_hand, 0, 1, 2)),
                    ("Simpson n=2", simpson(f_hand, 0, 1, 2)), ("Gauss 2 pontos", gauss_legendre(f_hand, 0, 1, 2)),
                    ("Gauss 3 pontos", gauss_legendre(f_hand, 0, 1, 3)),
                    ("Gauss 4 pontos", gauss_legendre(f_hand, 0, 1, 4))]:
    print(f"{nome:15s} = {valor:.6f}   erro = {valor - I_ref:+.1e}")

## 5. Explore

### 5.1 Erro em função do custo

O custo de uma integração é o número de avaliações de $f$. Para $\int_0^1 \frac{4}{1 + x^2}\,dx = \pi$:

In [ ]:
f_pi = lambda x: 4 / (1 + x**2)
ns = 2 ** np.arange(1, 11)
dados = {
    "trapézio": ([n + 1 for n in ns], [abs(trapezio(f_pi, 0, 1, n) - np.pi) for n in ns]),
    "Simpson": ([n + 1 for n in ns], [abs(simpson(f_pi, 0, 1, n) - np.pi) for n in ns]),
    "Gauss-Legendre": (list(range(1, 13)), [abs(gauss_legendre(f_pi, 0, 1, n) - np.pi) for n in range(1, 13)]),
}
rr = romberg(f_pi, 0, 1, tol=1e-15, max_niveis=8)
dados["Romberg"] = ([2**i + 1 for i in range(len(rr.historico))], [abs(v - np.pi) for v in rr.historico])
cores = {"trapézio": CORES["erro"], "Simpson": CORES["aproximacao"], "Gauss-Legendre": CORES["destaque"],
         "Romberg": CORES["secundaria"]}
fig = go.Figure()
for nome, (nav, err) in dados.items():
    err = np.maximum(err, 1e-17)
    fig.add_trace(go.Scatter(x=nav, y=err, mode="lines+markers", name=nome, line=dict(color=cores[nome])))
fig.update_xaxes(type="log", title="avaliações de f")
fig.update_yaxes(type="log", title="erro absoluto", exponentformat="power")
fig.update_layout(width=780, height=460, title="Integral de 4/(1 + x²) em [0, 1]", legend=dict(x=0.75, y=0.98))
mostrar(fig)

**O que notar:** em escala log-log, o trapézio e Simpson são retas de inclinação $-2$ e $-4$. Gauss e Romberg **curvam para baixo**: o erro cai mais depressa que qualquer potência. Gauss chega à precisão de máquina com 12 avaliações, e Romberg com pouco mais de cem, enquanto o trapézio, com mil avaliações, ainda erra na sétima casa. Para funções suaves que você pode avaliar onde quiser, regras de ordem alta valem muito a pena.


### 5.2 Nós e pesos de Gauss-Legendre

Arraste o slider para mudar o número de pontos $n$. Em cima, o polinômio de Legendre $P_n$ e suas raízes, que são os nós; embaixo, o peso de cada nó.

In [ ]:
tt = np.linspace(-1, 1, 500)

def quadro_nos(n):
    t, w = np.polynomial.legendre.leggauss(n)
    Pn = np.polynomial.legendre.Legendre.basis(n)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.55, 0.45])
    fig.add_trace(go.Scatter(x=tt, y=Pn(tt), mode="lines", line=dict(color=CORES["exata"], width=2),
                             name="P_n(t)"), 1, 1)
    fig.add_trace(go.Scatter(x=t, y=np.zeros(n), mode="markers", marker=dict(color=CORES["destaque"], size=10),
                             name="raízes = nós"), 1, 1)
    # pesos como hastes verticais
    xs, ys = [], []
    for ti, wi in zip(t, w):
        xs += [ti, ti, None]
        ys += [0, wi, None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=CORES["aproximacao"], width=3),
                             showlegend=False, hoverinfo="skip"), 2, 1)
    fig.add_trace(go.Scatter(x=t, y=w, mode="markers", marker=dict(color=CORES["aproximacao"], size=9),
                             name="pesos w_i"), 2, 1)
    fig.update_xaxes(range=[-1.05, 1.05])
    fig.update_xaxes(title="t", row=2, col=1)
    fig.update_yaxes(title="P_n(t)", range=[-1.1, 1.1], row=1, col=1)
    fig.update_yaxes(title="peso", range=[0, 1.05], row=2, col=1)
    fig.update_layout(width=780, height=560, legend=dict(orientation="h", y=-0.15),
                      title=f"n = {n}: menor distância de um nó à ponta = {numero(1 - t.max(), 4)}, "
                            f"soma dos pesos = {numero(w.sum(), 12)}")
    return fig

quadros = [(str(n), quadro_nos(n)) for n in range(1, 13)]
mostrar(com_controle(quadros, rotulo="n = ", ativo=4, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** os nós nunca tocam as pontas, mas se **acumulam** perto delas à medida que $n$ cresce (a distância à ponta cai como $1/n^2$); os pesos ficam menores ali, para compensar. Todos os pesos são positivos e a soma é sempre 2. É a mesma distribuição dos nós de Chebyshev, que evita o fenômeno de Runge na interpolação (capítulo 6): nós igualmente espaçados são a pior escolha também para integrar.

### 5.3 Gauss composto contra trapézio e Simpson

A seção 5.1 aumentou $n$ num intervalo só. Agora fixamos $n = 2$ ou $n = 3$ pontos e dividimos $[0, 1]$ em $m$ subintervalos, como se faz com o trapézio e Simpson. O integrando é o da seção 4, $e^{-x^2}$, cuja integral exata é $\frac{\sqrt\pi}{2}\operatorname{erf}(1)$.

(Por que não $4/(1 + x^2)$, da seção 5.1? Para ele, $f'''(0) = f'''(1) = 0$, e o primeiro termo do erro de Simpson, proporcional a $f'''(b) - f'''(a)$, some: Simpson ganharia duas ordens "de sorte". Um bom teste de convergência evita esses acasos.)

In [ ]:
from cnlivro.estilo import reta_de_ordem
from scipy.special import erf
f_g, I_g = lambda x: np.exp(-x**2), np.sqrt(np.pi) / 2 * erf(1.0)
ms = 2 ** np.arange(0, 9)
series = {
    "trapézio": ([2 * m + 1 for m in ms], [abs(trapezio(f_g, 0, 1, 2 * m) - I_g) for m in ms], CORES["erro"]),
    "Simpson": ([2 * m + 1 for m in ms], [abs(simpson(f_g, 0, 1, 2 * m) - I_g) for m in ms], CORES["aproximacao"]),
    "Gauss 2 pontos composto": ([2 * m for m in ms], [abs(gauss_legendre(f_g, 0, 1, 2, m) - I_g) for m in ms],
                                CORES["destaque"]),
    "Gauss 3 pontos composto": ([3 * m for m in ms], [abs(gauss_legendre(f_g, 0, 1, 3, m) - I_g) for m in ms],
                                CORES["secundaria"]),
}
fig = go.Figure()
for nome, (nav, err, cor) in series.items():
    nav, err = np.array(nav), np.array(err)
    visivel = err > 2e-15                            # abaixo disso, só arredondamento
    fig.add_trace(go.Scatter(x=nav[visivel], y=err[visivel], mode="lines+markers", name=nome, line=dict(color=cor)))
for p, x0, y0, x1 in [(2, 6, 5e-2, 300), (4, 10, 3e-5, 300), (6, 8, 1e-8, 60)]:
    # retas de referência com inclinação −p, acima de cada curva
    y1 = y0 * (x1 / x0) ** -p
    fig.add_trace(reta_de_ordem(x0, x1, y1, -p))
    fig.add_annotation(x=np.log10(x1), y=np.log10(y1), text=f"inclinação −{p}", showarrow=False,
                       xanchor="left", xshift=4, font=dict(color=CORES["referencia"], size=12))
fig.update_xaxes(type="log", title="avaliações de f")
fig.update_yaxes(type="log", title="erro absoluto", exponentformat="power", range=[-16.5, 0])
fig.update_layout(width=780, height=480, title="Refinamento h: m subintervalos com regras de 2 e 3 pontos",
                  legend=dict(x=0.02, y=0.02, xanchor="left", yanchor="bottom"))
mostrar(fig)

for nome, (nav, err, _) in series.items():
    nav, err = np.array(nav, float), np.array(err)
    ok = err > 1e-14                                 # longe do arredondamento
    print(f"{nome:24s} ordem observada (em avaliações): {-ordem_observada(nav[ok][-3:], err[ok][-3:]):.2f}")

**O que notar:** as curvas são retas com inclinações $-2$, $-4$, $-4$ e $-6$, como prevê {eq}`eq-erro-gauss` ($h^{2n}$). Gauss com 2 pontos tem a ordem de Simpson e fica um pouco abaixo dele: com o mesmo número de avaliações, erra cerca de 2/3 do que Simpson erra (são os fatores $1/4320$ e $1/2880$ da fórmula do erro). A vantagem real aparece com 3 pontos: Gauss ganha duas ordens com o custo de Simpson e chega a $10^{-13}$ com menos de 50 avaliações, quando o trapézio, com 500, ainda erra na sétima casa. Com integrandos caros (uma simulação inteira por avaliação), essa diferença é o que decide se a conta cabe no prazo.

### 5.4 A surpresa do trapézio: funções periódicas

Para integrar uma função **periódica** num período inteiro, o trapézio, a regra mais simples, é a melhor de todas:

In [ ]:
f_per = lambda x: 1 / (2 + np.cos(x))                  # ∫₀^{2π} = 2π/√3
exato_per = 2 * np.pi / np.sqrt(3)
ns = np.array([4, 8, 12, 16, 24, 32, 48, 64])
e_trap = [abs(trapezio(f_per, 0, 2 * np.pi, n) - exato_per) for n in ns]
e_simp = [abs(simpson(f_per, 0, 2 * np.pi, n) - exato_per) for n in ns]
fig = go.Figure()
fig.add_trace(go.Scatter(x=ns, y=np.maximum(e_trap, 1e-17), mode="lines+markers", name="trapézio",
                         line=dict(color=CORES["erro"])))
fig.add_trace(go.Scatter(x=ns, y=np.maximum(e_simp, 1e-17), mode="lines+markers", name="Simpson",
                         line=dict(color=CORES["aproximacao"])))
fig.update_xaxes(title="número de subintervalos n")
fig.update_yaxes(type="log", title="erro absoluto", exponentformat="power")
fig.update_layout(width=760, height=420, title="Integral de 1/(2 + cos x) num período (eixo x linear)")
mostrar(fig)

**O que notar:** com o eixo $n$ **linear**, o erro do trapézio é uma reta: ele cai **exponencialmente**, e chega a $10^{-16}$ com 32 pontos, enquanto Simpson ainda está em $10^{-9}$. Numa função periódica, todos os termos de Euler-Maclaurin se cancelam entre as duas pontas. Valores eficazes de correntes, potências médias em ciclos de motores e coeficientes de Fourier são integrais desse tipo: aí, o trapézio é a regra certa, e "melhorar" para Simpson piora.

### 5.5 Onde uma rotina adaptativa trabalha

Arraste o slider para mudar a tolerância do Simpson adaptativo aplicado a uma função com dois picos.

In [ ]:
f_picos = lambda x: 1 / ((x - 0.3) ** 2 + 0.01) + 1 / ((x - 0.9) ** 2 + 0.04) - 6
exato_picos = integrate.quad(f_picos, 0, 1, epsabs=1e-13)[0]
xx = np.linspace(0, 1, 600)

def quadro_adapt(tol):
    r = simpson_adaptativo(f_picos, 0.0, 1.0, tol=tol)
    bordas = sorted({c_ for iv in r.intervalos for c_ in iv})
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xx, y=f_picos(xx), mode="lines", name="f(x)", line=dict(color=CORES["exata"])))
    fig.add_trace(go.Scatter(x=bordas, y=np.full(len(bordas), -15), mode="markers", name="bordas dos subintervalos",
                             marker=dict(symbol="line-ns-open", size=14, color=CORES["aproximacao"])))
    fig.update_xaxes(title="x")
    fig.update_yaxes(title="f(x)", range=[-22, 100])
    fig.update_layout(width=780, height=430, legend=dict(x=0.6, y=0.98),
                      title=f"tol = {tol:.0e}: {r.avaliacoes} avaliações, erro = {abs(r.valor - exato_picos):.1e}"
                      .replace(".", ","))
    return fig

tols = [1e-2, 1e-4, 1e-6, 1e-8]
quadros = [(f"{t_:.0e}", quadro_adapt(t_)) for t_ in tols]
mostrar(com_controle(quadros, rotulo="tolerância = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** as bordas se acumulam nos dois picos, onde a curvatura é grande, e ficam esparsas no resto. Cada fator 100 na tolerância custa um fator de cerca de 3 nas avaliações, como esperado para ordem 4 ($100^{1/4} \approx 3{,}2$). Uma regra com passo uniforme precisaria do passo mais fino em todo o intervalo.

### 5.6 Monte Carlo contra a maldição da dimensão

Para $\int_{[0,1]^d} \prod_i \cos x_i \, d\mathbf{x} = (\sin 1)^d$, comparamos o trapézio em grade ($n$ pontos por eixo, $N = n^d$ avaliações) com Monte Carlo ($N$ pontos sorteados), em 1 e em 6 dimensões.

In [ ]:
rng = np.random.default_rng(0)
fig = make_subplots(rows=1, cols=2, subplot_titles=["d = 1", "d = 6"], horizontal_spacing=0.1)
for col, dim in [(1, 1), (2, 6)]:
    exato_d = np.sin(1.0) ** dim
    n_eixo = np.unique(np.round(np.logspace(0.3, 6 / dim, 12)).astype(int))
    N_grade = n_eixo**dim
    # a integral de um produto, numa grade produto, é o produto das integrais 1D
    e_grade = [abs(trapezio(np.cos, 0, 1, n - 1) ** dim - exato_d) for n in n_eixo if n > 1]
    fig.add_trace(go.Scatter(x=N_grade[n_eixo > 1], y=e_grade, mode="lines+markers", name="trapézio em grade",
                             line=dict(color=CORES["erro"]), showlegend=(col == 1)), 1, col)
    N_mc = np.logspace(2, 6, 9).astype(int)
    e_mc, barra = [], []
    for N in N_mc:
        r = monte_carlo(lambda p: np.prod(np.cos(p), axis=1), [(0, 1)] * dim, N, rng)
        e_mc.append(abs(r.valor - exato_d))
        barra.append(r.erros[0])
    fig.add_trace(go.Scatter(x=N_mc, y=e_mc, mode="markers", name="Monte Carlo (erro real)",
                             marker=dict(color=CORES["aproximacao"], size=8), showlegend=(col == 1)), 1, col)
    fig.add_trace(go.Scatter(x=N_mc, y=barra, mode="lines", name="Monte Carlo (erro padrão estimado)",
                             line=dict(color=CORES["aproximacao"], dash="dash"), showlegend=(col == 1)), 1, col)
fig.update_xaxes(type="log", title="avaliações N", exponentformat="power")
fig.update_yaxes(type="log", title="erro", exponentformat="power")
fig.update_layout(width=980, height=440, legend=dict(orientation="h", y=-0.22))
mostrar(fig)

**O que notar:** em uma dimensão, o trapézio ganha de longe (inclinação $-2$ contra $-1/2$). Em seis dimensões, o trapézio cai só como $N^{-1/3}$, e Monte Carlo, com a mesma inclinação $-1/2$ de antes, empata ou ganha. Repare também que o erro real de Monte Carlo oscila em torno do **erro padrão estimado** {eq}`eq-mc`: a barra de erro de Monte Carlo é uma estimativa estatística, e é ela que se deve relatar junto com o resultado. Em dimensões ainda maiores, como no cálculo de probabilidades de falha com dezenas de variáveis incertas (apêndice A), Monte Carlo é praticamente a única opção.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Calcule o tempo de esvaziamento do tanque pela integral de Torricelli, usando a regra de Simpson composta com 1000 subintervalos. Estime o erro comparando com 500 subintervalos e diga quantos algarismos do resultado são confiáveis."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
def tempo_esvaziamento(n=1000):
    h = np.linspace(0.0, H, n + 1)
    with np.errstate(divide="ignore"):
        y = area(h) / (k * np.sqrt(h))
    y[0] = 0.0          # em h = 0 o integrando é infinito; a contribuição de um ponto é desprezível
    passo = H / n
    return passo / 3 * (y[0] + 4 * y[1:-1:2].sum() + 2 * y[2:-1:2].sum() + y[-1])

T_1000 = tempo_esvaziamento(1000)
T_500 = tempo_esvaziamento(500)
erro_est = abs(T_1000 - T_500) / 15          # Richardson para Simpson, ordem 4
print(f"tempo de esvaziamento: {T_1000 / 60:.3f} min")
print(f"erro estimado: {erro_est:.2f} s  ->  resultado confiável até {erro_est / 60:.3f} min")

O código é limpo, a fórmula de Simpson está correta, e a estimativa de erro usa Richardson, como na seção 3.2. O resultado, 63,14 min com erro de cerca de 1 s, parece muito confiável.

### 6.2 Verificação

**Contra o valor exato.** Este tanque tem primitiva:

In [ ]:
print(f"exato:  {T_exato / 60:.3f} min")
print(f"agente: {T_1000 / 60:.3f} min;  erro real = {T_exato - T_1000:.0f} s,  erro estimado pelo agente = {erro_est:.1f} s")

O erro real é de cerca de 40 s, **trinta vezes** maior que o estimado. Sem a primitiva, que um tanque real não tem, como perceber?

**Pela ordem observada.** A estimativa de Richardson supõe que o erro cai como $h^4$. Isso pode ser **medido**, refinando a malha:

In [ ]:
ns = np.array([100, 1000, 10_000, 100_000])
valores = np.array([tempo_esvaziamento(n) for n in ns])
for n, v in zip(ns, valores):
    print(f"n = {n:>6d}: T = {v / 60:.4f} min")
dif = np.abs(np.diff(valores))
print("diferenças sucessivas (s):", np.round(dif, 2))
print(f"ordem observada pelas diferenças: {np.log10(dif[0] / dif[1]):.2f} e {np.log10(dif[1] / dif[2]):.2f}")

A cada fator 10 em $n$, o resultado muda por um fator de só $10^{0{,}5} \approx 3$: a ordem é $1/2$, não 4. Com essa ordem, o fator $1/15$ da estimativa do agente está errado, e com ele a confiança no resultado.

### 6.3 Por quê

A fórmula de erro de Simpson {eq}`eq-simpson` contém $f^{(4)}$, que é infinita em $h = 0$ para um integrando com $1/\sqrt{h}$. E a "correção" `y[0] = 0` não é inofensiva: a área sob o pico do primeiro subintervalo, $\int_0^{\Delta h} \frac{A_0}{k\sqrt{h}}\,dh = \frac{2A_0\sqrt{\Delta h}}{k}$, é proporcional a $\sqrt{\Delta h}$, e é essencialmente ela que falta. Daí a ordem $1/2$. Refinar não resolve: com $n = 100\,000$ o erro ainda passa de 4 s, e para errar menos de 1 s seriam precisos cerca de 2 milhões de subintervalos.

### 6.4 A correção: mude a variável, não o passo

Com $h = u^2$, temos $dh = 2u\,du$ e $\sqrt{h} = u$:

$$
T = \int_0^{\sqrt{H}} \frac{2A(u^2)}{C_d\,a\sqrt{2g}}\,du .
$$

O $\sqrt{h}$ do denominador desaparece. O novo integrando é suave; aqui, é até um polinômio de grau 4 em $u$, que Gauss com 3 pontos integra **exatamente**:

In [ ]:
integrando_u = lambda u: 2 * area(u**2) / k
for n in [2, 3, 5]:
    T_g = gauss_legendre(integrando_u, 0.0, np.sqrt(H), n)
    print(f"Gauss com {n} pontos, depois de h = u²: {T_g / 60:.6f} min  (erro {T_g - T_exato:+.1e} s)")
print(f"Simpson n = 1000, depois de h = u²:      {simpson(integrando_u, 0.0, np.sqrt(H), 1000) / 60:.6f} min")
T_quad, est_quad = integrate.quad(integrando, 0.0, H)
print(f"scipy.integrate.quad na forma original:  {T_quad / 60:.6f} min  (erro estimado {est_quad:.0e} s)")

Três avaliações do integrando bastam. O `quad` também acerta, porque o QUADPACK detecta e extrapola singularidades nas pontas, mas a mudança de variável funciona com qualquer regra e com uma tabela de arqueação medida.

> **Lição.** Uma estimativa de erro por Richardson é tão boa quanto a ordem que ela supõe. **Meça a ordem** refinando a malha antes de confiar numa barra de erro. Quando o integrando tem uma singularidade, **mude a variável** para removê-la: é mais barato e mais preciso que qualquer refinamento. E desconfie de "correções" como `y[0] = 0`, que escondem um problema em vez de resolvê-lo.

## 7. Quando falha

### 7.1 Um pico estreito num intervalo longo

Um sensor de força registra o impacto de uma peça durante 1 s. O pulso dura alguns milissegundos e acontece em $t = 700$ ms. O **impulso** é a integral da força:

In [ ]:
forca = lambda t_ms: 2000 * np.exp(-(((t_ms - 700) / 2) ** 2))           # N, t em ms
impulso_exato = 2000 * 2 * np.sqrt(np.pi) / 1000                          # N·s
val, est = integrate.quad(forca, 0, 1000)
print(f"quad em [0, 1000] ms: {val / 1000:.3e} N·s (erro estimado {est / 1000:.0e})   exato: {impulso_exato:.3f} N·s")
val2, _ = integrate.quad(forca, 0, 1000, points=[700])
print(f"quad avisado do pico (points=[700]): {val2 / 1000:.3f} N·s")

O `quad` devolve praticamente zero, com uma estimativa de erro minúscula e **sem nenhum aviso**. Os primeiros 21 pontos de Gauss-Kronrod caíram todos longe do pulso, onde a força é zero, e a rotina concluiu que a função é zero. Uma rotina adaptativa só refina onde **vê** dificuldade. Se você sabe onde estão os picos, diga a ela (`points=`), ou divida o intervalo. Desenhe sempre o integrando antes de integrar.

### 7.2 Descontinuidades

Uma carga distribuída que muda bruscamente num ponto (uma parede sobre uma laje, por exemplo) tem um salto. Com o salto fora dos nós, toda regra de Newton-Cotes perde a ordem:

In [ ]:
carga = lambda x: np.where(x < 1 / 3, 2.0, 5.0)        # kN/m, salto em x = 1/3
exato_carga = 2.0 / 3 + 5.0 * 2 / 3
ns = np.array([10, 20, 40, 80, 160])
e_t = [abs(trapezio(carga, 0, 1, n) - exato_carga) for n in ns]
e_s = [abs(simpson(carga, 0, 1, n) - exato_carga) for n in ns]
print(f"ordem observada: trapézio {ordem_observada(1 / ns, e_t):.2f}, Simpson {ordem_observada(1 / ns, e_s):.2f}")
partes = simpson(carga, 0, 1 / 3 - 1e-15, 2) + simpson(carga, 1 / 3, 1, 2)
print(f"dividindo no salto, Simpson com 2 + 2 subintervalos: erro = {abs(partes - exato_carga):.1e}")

As duas regras caem para ordem 1, com erros que oscilam conforme a posição do salto entre os nós. Dividir a integral no ponto de descontinuidade devolve a ordem e, aqui, a exatidão.

### 7.3 Dados com ruído

Quando os valores vêm de medidas, a regra de ordem alta deixa de ajudar. O erro do ruído na integral é da ordem de $\sigma\sqrt{(b - a)h}$, independentemente da regra. Repetindo 300 vezes um "ensaio" com ruído de 5 % sobre $\int_0^\pi \sin x\,dx = 2$:

In [ ]:
rng = np.random.default_rng(5)
sig = 0.05                                                   # ruído de 5 % do pico
for n in [10, 100, 1000]:
    x = np.linspace(0, np.pi, n + 1)
    e_t, e_s = [], []
    for _ in range(300):                                     # 300 repetições do "ensaio"
        y = np.sin(x) + rng.normal(0, sig, x.size)
        e_t.append(trapezio_dados(x, y) - 2)
        e_s.append(integrate.simpson(y, x=x) - 2)
    print(f"n = {n:4d}: erro típico (RMS) trapézio = {np.sqrt(np.mean(np.square(e_t))):.4f}, "
          f"Simpson = {np.sqrt(np.mean(np.square(e_s))):.4f};  previsão σ√((b−a)h) = {sig * np.sqrt(np.pi * np.pi / n):.4f}")

Com ruído, Simpson e o trapézio erram por quantidades parecidas, e mais pontos ajudam só como $\sqrt{h}$, uma média estatística. Integrar dados ruidosos é bem condicionado (o ruído se cancela em parte), ao contrário de derivar (capítulo 10), mas a ordem das regras não importa mais.


### 7.4 Gauss não é mágica: integrandos que não são suaves

A convergência exponencial de Gauss depende das derivadas altas em {eq}`eq-erro-gauss`. Se $f$ tem uma quina ou uma raiz quadrada, essas derivadas não existem, e Gauss vira uma regra como as outras. Três integrandos em $[0, 1]$, com $n$ crescente num intervalo só:

In [ ]:
casos = {
    "e^x (suave)": (np.exp, np.e - 1),
    "|x − 1/3| (quina)": (lambda x: np.abs(x - 1 / 3), (1 / 3) ** 2 / 2 + (2 / 3) ** 2 / 2),
    "√x (raiz na ponta)": (np.sqrt, 2 / 3),
}
ns_g = np.array([2, 4, 8, 16, 32, 64])
for nome, (f_c, ex) in casos.items():
    e = np.array([abs(gauss_legendre(f_c, 0, 1, n) - ex) for n in ns_g])
    linha = "  ".join(f"{v:.0e}" for v in e)
    ok = e > 1e-14
    ordem = f"ordem em n ≈ {ordem_observada(1 / ns_g[ok], e[ok]):.1f}" if ok.sum() >= 3 else "precisão de máquina"
    print(f"{nome:20s} {linha}   ({ordem})")
print("n:", ns_g)
print(f"√x depois de x = u² (integrando 2u², polinômio): erro com 2 pontos = "
      f"{abs(gauss_legendre(lambda u: 2 * u**2, 0, 1, 2) - 2 / 3):.0e}")

Para $e^x$, 8 pontos já dão a precisão de máquina. Para a quina, o erro cai só como $n^{-2}$, aproximadamente, a ordem do trapézio, e oscila conforme a quina cai perto ou longe de um nó. Para $\sqrt{x}$, cai como $n^{-3}$, aproximadamente. As soluções são as mesmas das seções 6.4 e 7.2: **dividir** o intervalo na quina (Gauss composto com uma borda em $x = 1/3$ volta a ser exato para este caso), ou **mudar a variável** para remover a raiz. Uma quadratura de ordem alta só é tão boa quanto a suavidade do integrando permite.

## Resumo

| Método | Exato para | Erro | Quando usar |
| --- | --- | --- | --- |
| Trapézio composto | retas | $O(h^2)$; exponencial para periódicas | dados tabelados; funções periódicas num período |
| Simpson 1/3 composto | cúbicas | $O(h^4)$ | dados igualmente espaçados e lisos |
| Romberg | | $O(h^{2k})$, cada nível + 2 ordens | funções suaves que se pode avaliar |
| Gauss-Legendre ($n$ pontos) | grau $2n - 1$ | exponencial para funções analíticas | funções suaves; integrandos caros |
| Gauss composto ($m$ pedaços) | grau $2n - 1$ em cada pedaço | $O(h^{2n})$ | funções com quinas conhecidas (divida nelas) |
| Gauss com peso (Hermite, Laguerre...) | peso × grau $2n - 1$ | exponencial | intervalos infinitos; médias sob a normal |
| Adaptativa (`quad`) | | controlado pela tolerância | padrão; diga onde estão picos e quinas |
| Mudança de variável | | remove singularidades | $1/\sqrt{x}$, intervalos infinitos |
| Monte Carlo | | $O(N^{-1/2})$ em qualquer dimensão | muitas dimensões; relate o erro padrão |

## Exercícios

**Conceituais (sem computador)**

1. Deduza a regra 1/3 de Simpson integrando a parábola de Lagrange pelos três nós, e mostre que ela é exata para $x^3$.
2. Encontre os nós e pesos da quadratura de Gauss com 2 pontos impondo que ela seja exata para $1, t, t^2, t^3$, **sem** supor simetria: mostre que $t_2 = -t_1$ sai das próprias equações.
3. Mostre que nenhuma regra com $n$ nós é exata para todos os polinômios de grau $2n$ (use $\prod_i (t - t_i)^2$).
4. Calcule à mão $\int_1^3 dx/x$ com Gauss de 2 e de 3 pontos e compare com $\ln 3$. Quantas avaliações o trapézio composto precisaria para errar menos que Gauss com 3 pontos? (Use {eq}`eq-trapezio`.)
5. Verifique que as raízes de $P_3 = (5t^3 - 3t)/2$ e a fórmula $w_i = 2/[(1 - t_i^2) P_3'(t_i)^2]$ dão os nós e pesos de {eq}`eq-gauss3`.
6. Mostre que a primeira coluna de Romberg, $(4T_{2n} - T_n)/3$, é a regra de Simpson com $2n$ subintervalos.
7. Para $\int_0^\infty e^{-x}/(1 + x^2)\,dx$, proponha uma mudança de variável que leve a um intervalo finito com integrando limitado.

**Verificação (com agente)**

8. Peça a um agente o tempo de esvaziamento de um tanque a partir de uma tabela de arqueação com 12 níveis irregulares (invente uma). Verifique se o código trata a singularidade em $h = 0$ e o espaçamento irregular (sugestão: interpole $A$ em função de $u = \sqrt{h}$).
9. Peça a um agente a estimativa por Monte Carlo do volume de uma esfera de raio 1 em 10 dimensões, com barra de erro. Verifique com a fórmula exata e repita com 20 sementes diferentes: a fração de vezes em que o valor exato cai dentro de $\pm 2$ erros padrão é próxima de 95 %?
10. Peça a um agente o valor eficaz (RMS) de uma corrente distorcida, $i(t) = 10\sin\omega t + 3\sin 3\omega t + \sin 5\omega t$, a partir de 64 amostras num período. Compare trapézio e Simpson com o valor exato $\sqrt{(100 + 9 + 1)/2}$ e explique o resultado com a seção 5.4.
11. Peça a um agente uma função que calcule os nós e pesos de Gauss-Legendre para qualquer $n$ **sem** usar `leggauss`: pela recorrência {eq}`eq-legendre` e pelo método de Newton, ou pelos autovalores da matriz de Jacobi. Verifique com os três testes da seção 3.3 e contra `leggauss` para $n = 5$, $20$ e $100$. Onde a precisão começa a se perder, e por quê?
12. Uma viga tem módulo de elasticidade incerto, $E$ normal com média 200 GPa e desvio 10 GPa, e a flecha é proporcional a $1/E$. Peça a um agente a flecha média por Gauss-Hermite com 3, 5 e 10 pontos e por Monte Carlo com $10^5$ sorteios. Compare custo e precisão.

**Projeto aberto**

13. **Esvaziamento com perda de carga.** Num tubo de descarga longo, a vazão depende do fator de atrito (capítulo 3), que depende da própria vazão. Monte a integral do tempo de esvaziamento com o fator de atrito calculado por Colebrook a cada nível e compare com Torricelli.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulos 21 e 22.
- P. J. Davis e P. Rabinowitz, *Methods of Numerical Integration*, 2ª ed., Academic Press, 1984.
- L. N. Trefethen e J. A. C. Weideman, "The exponentially convergent trapezoidal rule", *SIAM Review*, 56(3), 2014.
- G. H. Golub e J. H. Welsch, "Calculation of Gauss quadrature rules", *Mathematics of Computation*, 23(106), 1969.
- L. N. Trefethen, "Is Gauss quadrature better than Clenshaw-Curtis?", *SIAM Review*, 50(1), 2008.
- R. Piessens e outros, *QUADPACK: A Subroutine Package for Automatic Integration*, Springer, 1983.